# THÊM VLM vào CUỐI inference X-VLM (đẩy R@1 / mAP)

**Dán 3 cell dưới vào CUỐI notebook `aicity-inference-v1` của bạn** — ngay sau khi pipeline đã ghi
`answer.txt` (cell tạo `answer.txt`). Vẫn LÀ 1 luồng inference + evaluate như cũ, **chỉ thêm 1 bước
VLM ở cuối**: Qwen2-VL-2B nhìn **top-10 ảnh** của mỗi query rồi chọn ảnh đúng nhất → đưa lên **rank-1**.

**Vì sao chạy bằng `!python` (subprocess):** X-VLM dùng transformers cũ, Qwen cần transformers mới —
chạy tiến trình con thì Qwen nạp transformers mới **riêng**, KHÔNG đụng X-VLM đang chạy trong kernel.
→ Không cần tách notebook, không upload answer.txt.

**Chỉ cần thêm 1 input:** **Qwen2-VL-2B-Instruct** (Kaggle Models → search "Qwen2-VL-2B").
2B fp16 ~5GB → **1×T4 dư sức, KHÔNG quantize**. ~1-2s/query × ~1978 ≈ **~0.5-1h**, có RESUME.

In [ ]:
# [VLM-1] Tim answer.txt (notebook VUA tao), query files, GT, va model Qwen trong input
import glob, os
def _one(p):
    h = sorted(glob.glob(f"/kaggle/working/**/{p}", recursive=True)) + \
        sorted(glob.glob(f"/kaggle/input/**/{p}", recursive=True))
    return h[0] if h else None

ANSWER = _one("answer.txt")                       # /kaggle/working/outputs/answer.txt (pipeline vua ghi)
QJSON  = _one("query_text.json")
QIDX   = _one("query_index.txt")
GT     = _one("ground_truth.txt")
qcfg   = [p for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "qwen" in p.lower()]
QWEN   = os.path.dirname(qcfg[0]) if qcfg else "Qwen/Qwen2-VL-2B-Instruct"   # local -> khoi tai HF
assert ANSWER, "Khong thay answer.txt — chay cac cell pipeline o tren TRUOC (no ghi /kaggle/working/outputs/answer.txt)"
assert QJSON and QIDX and GT, f"Thieu: QJSON={QJSON} QIDX={QIDX} GT={GT}"
print("ANSWER :", ANSWER)
print("QWEN   :", QWEN, "(local)" if qcfg else "(tai tu HuggingFace)")

In [ ]:
%%writefile /kaggle/working/qwen_rerank.py
"""Qwen2-VL top-10 re-rank — chay nhu MOT TIEN TRINH CON o cuoi notebook inference X-VLM.

Vi sao subprocess: X-VLM pin transformers cu, Qwen2-VL can transformers moi -> khong chung 1
kernel. Chay `!python qwen_rerank.py ...` la tien trinh Python rieng -> import transformers moi
sach se, KHONG dung X-VLM da nap trong kernel.

Doc answer.txt pipeline VUA xuat (top-N image_id/query, thu tu = query_index), cho Qwen2-VL chon
anh khop nhat trong top-K -> dua len rank-1, ghi answer_vlm.txt, in mAP/R@1/R@5/R@10 truoc/sau.
Resumable (luu picks json) -> bi dung thi chay lai chay tiep.

    python qwen_rerank.py --answer /kaggle/working/outputs/answer.txt \
        --query-json query_text.json --query-index query_index.txt --gt ground_truth.txt \
        --image-root /kaggle/input --model <qwen_dir|HF id> --out /kaggle/working/outputs/answer_vlm.txt
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import random
import re
import time
from pathlib import Path

_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")


def load_queries(answer, query_json, query_index, gt, image_root, topk, skip_first_col=False):
    raw = open(query_json, encoding="utf-8").read().strip()
    recs = json.loads(raw) if raw.startswith("[") else [json.loads(l) for l in raw.splitlines() if l.strip()]
    cap_of = {str(r["query_index"]): r["caption"] for r in recs}
    qorder = [l.strip() for l in open(query_index, encoding="utf-8").read().strip().splitlines()]
    gts = [l.strip() for l in open(gt, encoding="utf-8").read().strip().splitlines()]
    lines = [l.split() for l in open(answer, encoding="utf-8").read().strip().splitlines()]
    if skip_first_col:
        lines = [t[1:] for t in lines]
    assert len(qorder) == len(gts) == len(lines), \
        f"lech dong: query_index={len(qorder)} gt={len(gts)} answer={len(lines)}"
    stem2path = {Path(p).stem: p for p in glob.glob(os.path.join(image_root, "**", "*"), recursive=True)
                 if p.lower().endswith(_EXTS)}
    queries = []
    for i, full in enumerate(lines):
        cands = full[:topk]
        queries.append(dict(full=full, candidates=cands,
                            paths=[stem2path[c] for c in cands if c in stem2path],
                            caption=cap_of.get(qorder[i], ""), gt=gts[i]))
    return queries, len(stem2path)


def rerank_metrics(queries, picks):
    """Pure (testable): before = X-VLM order; after = VLM-picked promoted to rank-1."""
    def before(q):
        return q["full"].index(q["gt"]) + 1 if q["gt"] in q["full"] else None

    def after(i, q):
        p = picks.get(str(i))
        new = ([p] + [x for x in q["full"] if x != p]) if p in q["full"] else q["full"]
        return new.index(q["gt"]) + 1 if q["gt"] in new else None

    def agg(ranks):
        rr = [1.0 / r if r else 0.0 for r in ranks]
        valid = [r for r in ranks if r]
        hit = lambda k: (sum(1 for r in valid if r <= k) / len(valid)) if valid else 0.0
        return dict(mAP=sum(rr) / len(rr) if rr else 0.0, R1=hit(1), R5=hit(5), R10=hit(10))

    return agg([before(q) for q in queries]), agg([after(i, q) for i, q in enumerate(queries)])


def new_order(full, picked):
    return ([picked] + [x for x in full if x != picked]) if picked in full else full


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--answer", required=True)
    ap.add_argument("--query-json", required=True)
    ap.add_argument("--query-index", required=True)
    ap.add_argument("--gt", required=True)
    ap.add_argument("--image-root", default="/kaggle/input")
    ap.add_argument("--model", required=True, help="local Qwen dir or HF id (Qwen/Qwen2-VL-2B-Instruct)")
    ap.add_argument("--out", default="/kaggle/working/outputs/answer_vlm.txt")
    ap.add_argument("--picks", default="/kaggle/working/vlm_picks.json")
    ap.add_argument("--topk", type=int, default=10)
    ap.add_argument("--max-pixels", type=int, default=512 * 28 * 28)
    ap.add_argument("--min-pixels", type=int, default=256 * 28 * 28)
    ap.add_argument("--quant", default="none", choices=["none", "8bit"])
    ap.add_argument("--no-shuffle", action="store_true")
    ap.add_argument("--skip-first-col", action="store_true", help="set if answer.txt lines start with a query id")
    args = ap.parse_args()

    queries, n_img = load_queries(args.answer, args.query_json, args.query_index, args.gt,
                                  args.image_root, args.topk, args.skip_first_col)
    print(f"queries={len(queries)} | images found={n_img} | topk={args.topk}")

    import torch
    from transformers import AutoProcessor
    try:
        from transformers import AutoModelForImageTextToText as VLMClass
    except ImportError:
        from transformers import Qwen2VLForConditionalGeneration as VLMClass
    from qwen_vl_utils import process_vision_info

    kw = dict(device_map="auto", torch_dtype=torch.float16)
    if args.quant == "8bit":
        from transformers import BitsAndBytesConfig
        kw = dict(device_map="auto", quantization_config=BitsAndBytesConfig(load_in_8bit=True))
    model = VLMClass.from_pretrained(args.model, **kw).eval()
    processor = AutoProcessor.from_pretrained(args.model, min_pixels=args.min_pixels, max_pixels=args.max_pixels)
    print("VLM ready | quant =", args.quant)

    picks = json.load(open(args.picks)) if Path(args.picks).exists() else {}
    random.seed(0)

    def prompt(n, cap):
        return ("Below are " + str(n) + " candidate images of a person, numbered 1 to " + str(n) +
                ", then a description. Pick the ONE image that best matches it. "
                "Reply with ONLY the number (1-" + str(n) + ").\nDescription: " + str(cap))

    t0, done0 = time.time(), len(picks)
    for i, q in enumerate(queries):
        if str(i) in picks:
            continue
        paths, cands = q["paths"], q["candidates"]
        if not paths:
            picks[str(i)] = cands[0] if cands else ""
            continue
        order = list(range(len(paths)))
        if not args.no_shuffle:
            random.shuffle(order)
        content = [{"type": "image", "image": paths[o]} for o in order]
        content.append({"type": "text", "text": prompt(len(paths), q["caption"])})
        msgs = [{"role": "user", "content": content}]
        try:
            text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            ii, vi = process_vision_info(msgs)
            inp = processor(text=[text], images=ii, videos=vi, padding=True, return_tensors="pt").to(model.device)
            with torch.no_grad():
                gen = model.generate(**inp, max_new_tokens=8, do_sample=False)
            out = processor.batch_decode(gen[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0]
            m = re.search(r"\d+", out)
            sel = int(m.group()) - 1 if m else 0
            sel = sel if 0 <= sel < len(paths) else 0
            picks[str(i)] = cands[order[sel]]
        except Exception as e:
            picks[str(i)] = cands[0]
            if i % 200 == 0:
                print("warn q", i, repr(e)[:120])
        if i % 50 == 0:
            json.dump(picks, open(args.picks, "w"))
            torch.cuda.empty_cache()
            el = (time.time() - t0) / 60
            eta = el / max(len(picks) - done0, 1) * (len(queries) - len(picks))
            print(f"{len(picks)}/{len(queries)}  {el:.1f}m  ETA ~{eta:.0f}m", flush=True)
    json.dump(picks, open(args.picks, "w"))

    B, A = rerank_metrics(queries, picks)
    print(f"\n{'':8s}{'mAP':>9}{'R@1':>9}{'R@5':>9}{'R@10':>9}")
    print(f"{'X-VLM':8s}{B['mAP']:9.4f}{B['R1']:9.4f}{B['R5']:9.4f}{B['R10']:9.4f}")
    print(f"{'+VLM':8s}{A['mAP']:9.4f}{A['R1']:9.4f}{A['R5']:9.4f}{A['R10']:9.4f}")
    print(f"{'delta':8s}{A['mAP']-B['mAP']:+9.4f}{A['R1']-B['R1']:+9.4f}{A['R5']-B['R5']:+9.4f}{A['R10']-B['R10']:+9.4f}")
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    with open(args.out, "w") as f:
        for i, q in enumerate(queries):
            f.write(" ".join(new_order(q["full"], picks.get(str(i)))) + "\n")
    print("\nwrote", args.out)


if __name__ == "__main__":
    main()


In [ ]:
# [VLM-3] Cai transformers MOI cho tien trinh con + chay VLM re-rank (subprocess -> khong dung X-VLM)
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
!pip -q install -U "transformers>=4.49.0" "accelerate>=0.34" qwen-vl-utils hf_transfer
# 2B fp16 (KHONG quantize). 1xT4 du. Neu doi sang 7B + 1 GPU thi them: --quant 8bit
cmd = (f'python /kaggle/working/qwen_rerank.py --answer "{ANSWER}" --query-json "{QJSON}" '
       f'--query-index "{QIDX}" --gt "{GT}" --image-root /kaggle/input --model "{QWEN}" '
       f'--out /kaggle/working/outputs/answer_vlm.txt --topk 10')
print(cmd)
!{cmd}
# Bang mAP/R@1/R@5/R@10 truoc-sau in ngay duoi. answer_vlm.txt = ban da day VLM len rank-1.

## Cách chạy
1. **Settings**: GPU **T4** (1 cái đủ cho 2B), Internet **ON**.
2. **Add Input**: giữ nguyên các dataset cũ của bạn **+ thêm `Qwen2-VL-2B-Instruct`** (Kaggle Models). KHÔNG upload answer.txt.
3. Chạy **toàn bộ notebook** như mọi khi → tới cuối, 3 cell VLM này chạy tiếp.
4. **Bị dừng?** Chạy lại — `qwen_rerank.py` đọc `vlm_picks.json` và chạy tiếp từ chỗ dở.

**Kết quả:** cell [VLM-3] in bảng `X-VLM vs +VLM` (delta) + ghi `/kaggle/working/outputs/answer_vlm.txt`.
Regime này R@10≈99% → VLM đẩy ảnh đúng từ top-2..5 lên rank-1 → **R@1 & mAP tăng mạnh**.

**Lưu ý:**
- `--image-root /kaggle/input` → script tự quét ảnh gallery trong input (khớp image_id).
- Nếu mỗi dòng `answer.txt` có **query-id ở đầu**, thêm cờ `--skip-first-col` vào cell [VLM-3].
- Không phụ thuộc HF nếu add Qwen qua Kaggle Models. **Không cắt bit** (2B fp16). Muốn 7B: đổi `--model` + `--quant 8bit` (1 GPU) hoặc bật T4×2.
- ⚠️ Số trên no-distractor + caption chi tiết → cao hơn đề thi thật (distractor + caption ngắn).